# 11 — `modulation_index`: a single MI value

The Tort modulation index quantifies how strongly the *amplitude*
of a high-frequency band is locked to the *phase* of a low-frequency
one. We feed it real `(pha, amp)` time series — phase in radians,
amplitude as the analytic-signal envelope — typically obtained via
`bandpass + hilbert`.

Expected shape: `(B, C, n_freqs, segments, samples)`.

In [1]:
import numpy as np
import torch
import scitex_dsp as dsp
%matplotlib inline

# A signal with strong theta-gamma PAC.
xx, tt, fs = dsp.demo_sig(sig_type="tensorpac", batch_size=1, n_chs=1,
                          n_segments=20, t_sec=2, fs=512)
print("input:", xx.shape)  # (B, C, segments, samples)

input: (1, 1, 20, 1024)


## Extract phase and amplitude

Bandpass + hilbert per band — phase from the slow band, amplitude
envelope from the fast band.

In [2]:
# Reshape (B, C, segments, samples) -> (B, C*segments, samples) for filt
B, C, S, T = xx.shape
flat = xx.reshape(B, C * S, T)

pha_band = np.asarray(dsp.filt.bandpass(flat, fs=fs, bands=[(6, 14)]))
amp_band = np.asarray(dsp.filt.bandpass(flat, fs=fs, bands=[(60, 120)]))
# bandpass adds a band axis: (B, C*S, n_bands, T) — drop it (n_bands=1)
pha_band = pha_band[..., 0, :]
amp_band = amp_band[..., 0, :]

# Hilbert: phase (radians) for the slow band, amplitude for the fast.
phase, _ = dsp.hilbert(pha_band)
_, amp   = dsp.hilbert(amp_band)
phase = np.asarray(phase); amp = np.asarray(amp)

# Reshape to (B, C, n_freqs=1, segments, samples).
phase = phase.reshape(B, C, S, T)[:, :, None]
amp   = amp.reshape(B, C, S, T)[:, :, None]
print("phase:", phase.shape, "range:", float(phase.min()), float(phase.max()))
print("amp  :", amp.shape, "min/max:", float(amp.min()), float(amp.max()))

phase: (1, 1, 1, 20, 1024) range: -3.141305923461914 3.141181468963623
amp  : (1, 1, 1, 20, 1024) min/max: 0.003249497851356864 2.4573380947113037


In [3]:
mi = dsp.modulation_index(torch.tensor(phase), torch.tensor(amp), n_bins=18)
print("MI shape:", tuple(mi.shape), "value:", float(mi.mean()))

MI shape: (1, 1, 1, 1) value: 0.010837221518158913


See `12_pac.ipynb` for the full pipeline that scans many phase- and amplitude-frequency combinations at once.